In [ ]:
import os
import sys
from pathlib import Path

# Set these before importing any `reva` modules. Update the placeholder paths
# to match your machine or shared Jupyter environment.
os.environ["REVA_DATA_ROOT"] = "/path/to/your/data/root"
os.environ["REVA_HF_CACHE_ROOT"] = "/path/to/your/hf_cache/root"
os.environ["REVA_CHECKPOINT_ROOT"] = "/path/to/your/checkpoints/root"
os.environ["REVA_EVAL_RESULTS_ROOT"] = "/path/to/your/eval_results/root"
os.environ["REVA_REGION_DATA_ROOT"] = "/path/to/your/region_data/root"
os.environ["REVA_DECONTAMINATION_ROOT"] = "/path/to/your/decontamination/root"
os.environ["REVA_GROUNDING_DINO_ROOT"] = "/path/to/your/groundingdino/root"
os.environ["REVA_VQAV2_ROOT"] = "/path/to/your/vqav2/root"
os.environ["REVA_TEST_IMAGES_ROOT"] = "/path/to/your/test_images/root"

hf_cache_root = Path(os.environ["REVA_HF_CACHE_ROOT"]).expanduser()
os.environ["HF_HOME"] = str(hf_cache_root)
os.environ["HF_HUB_CACHE"] = str(hf_cache_root / "hub")
os.environ["HF_DATASETS_CACHE"] = str(hf_cache_root / "datasets")
os.environ["TRANSFORMERS_CACHE"] = str(hf_cache_root / "hub")

for var_name in (
    "REVA_DATA_ROOT",
    "REVA_HF_CACHE_ROOT",
    "REVA_CHECKPOINT_ROOT",
    "REVA_EVAL_RESULTS_ROOT",
    "REVA_REGION_DATA_ROOT",
    "REVA_DECONTAMINATION_ROOT",
    "REVA_GROUNDING_DINO_ROOT",
    "REVA_VQAV2_ROOT",
    "REVA_TEST_IMAGES_ROOT",
    "HF_HOME",
    "HF_HUB_CACHE",
    "HF_DATASETS_CACHE",
    "TRANSFORMERS_CACHE",
):
    print(f"{var_name} = {os.environ.get(var_name)}")


def find_reva_project_root(start: Path) -> Path:
    override = os.environ.get("REVA_PROJECT_DIR")
    if override:
        return Path(override).expanduser().resolve()

    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "reva" / "evaluation.py").is_file() and (candidate / "reva" / "config.py").is_file():
            return candidate

    raise FileNotFoundError(
        "Could not locate the ReVA project root from the current working directory. "
        "Set REVA_PROJECT_DIR to your cloned repo path."
    )


PROJECT_ROOT = find_reva_project_root(Path.cwd())

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.chdir(PROJECT_ROOT)
print("Working directory:", Path.cwd())


# VQAv2 test-dev / test-standard pHash Decontamination

Remove curriculum samples whose images perceptually overlap **VQAv2 evaluation images**.

**Reference benchmark:** VQAv2 test-dev and test-standard both use **COCO test2015** images (`COCO_test2015_*.jpg`). One pHash scan against all test2015 JPGs covers **both** eval splits.

**Technique:** pHash Hamming distance ≤ 4 (same as POPE pipeline / `dataset.py`).

**Recommended input:** `curriculum_pope_clean.pkl` if you already ran POPE decontamination (chain purges). Falls back to building the full curriculum if that pickle is missing.

**Outputs:**
- `curriculum_vs_vqav2_test2015_phash.json` — full match log
- `phash_matches_curriculum_vqav2_report.csv` — CSV audit trail
- `curriculum_pope_vqav2_clean.pkl` — filtered sample list ready for Stage 2 training

In [ ]:
# Core decontamination + curriculum loaders (run once per environment)
!pip install imagehash pillow "numpy<2.0" tqdm matplotlib pandas pyarrow datasets huggingface_hub img2dataset --quiet

## 0. Download VQAv2 eval assets (run once)

Downloads **COCO test2015** images (~12 GB) and VQAv2 **test-dev + test-standard** question JSONs (same zip).

Skip sections that already exist on disk.

In [ ]:
%%bash
set -euo pipefail

VQAV2_DIR="${REVA_VQAV2_ROOT:-$HOME/reva-data/vqav2}"
LOG_DIR="${REVA_DOWNLOAD_LOG_ROOT:-$HOME/reva-data/download_logs}"
mkdir -p "$VQAV2_DIR" "$LOG_DIR"

step()     { echo "[$(date +%H:%M:%S)] $1"; }
done_msg() { echo "[$(date +%H:%M:%S)] done: $1"; }
skip_msg() { echo "[$(date +%H:%M:%S)] skip: $1 (already present)"; }

cd "$VQAV2_DIR"

# test-dev + test-standard question JSONs (same zip)
if [ ! -f v2_OpenEnded_mscoco_test-dev2015_questions.json ] || [ ! -f v2_OpenEnded_mscoco_test2015_questions.json ]; then
  step "VQAv2 test question JSONs (test-dev + test-std) ..."
  wget -q -c -O v2_Questions_Test_mscoco.zip https://s3.amazonaws.com/cvmlp/vqa/mscoco/vqa/v2_Questions_Test_mscoco.zip
  unzip -qo v2_Questions_Test_mscoco.zip && rm -f v2_Questions_Test_mscoco.zip
  done_msg "VQAv2 test question JSONs"
else skip_msg "VQAv2 test question JSONs"; fi

# COCO test2015 images — reference for both splits
if [ ! -d test2015 ] || [ -z "$(find test2015 -maxdepth 1 -name '*.jpg' 2>/dev/null | head -1)" ]; then
  step "COCO test2015 images (~12 GB) ..."
  wget -q -c -O test2015.zip http://images.cocodataset.org/zips/test2015.zip
  unzip -qo test2015.zip && rm -f test2015.zip
  done_msg "COCO test2015"
else skip_msg "COCO test2015"; fi

echo "[$(date +%H:%M:%S)] VQAv2 eval assets ready under $VQAV2_DIR"

## 1. Setup

In [ ]:
import os
import json
import csv
import pickle
import random
from pathlib import Path
from collections import Counter

import matplotlib.pyplot as plt
from PIL import Image

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'
os.environ['HF_HOME'] = os.environ.get('REVA_HF_CACHE_ROOT') or os.path.expanduser('~/reva-data/hf_cache')

from reva.config import ProjectionAConfig
from reva.dataset import (
    load_coco_detection_samples,
    load_refcoco_samples,
    load_visual_genome_samples,
    load_grit_samples,
    collect_vqav2_test2015_paths,
    load_vqav2_eval_image_ids,
    generate_curriculum_decontamination_log,
    filter_curriculum_from_log,
    HAMMING_THRESH,
)

config = ProjectionAConfig()
DECONTAM_DIR = Path(os.environ.get('REVA_DECONTAMINATION_ROOT') or os.path.expanduser('~/reva-data/decontamination'))
VQAV2_DIR = Path(os.environ.get('REVA_VQAV2_ROOT') or os.path.expanduser('~/reva-data/vqav2'))
DECONTAM_DIR.mkdir(parents=True, exist_ok=True)

POPE_CLEAN_PKL = DECONTAM_DIR / 'curriculum_pope_clean.pkl'

print('Config ready.')
print(f'data_dir: {config.data_dir}')
print(f'pHash Hamming threshold: {HAMMING_THRESH}')

## 2. Load curriculum

Prefer **`curriculum_pope_clean.pkl`** (chain after POPE). Otherwise build the full combined curriculum.

In [ ]:
if POPE_CLEAN_PKL.exists():
    with open(POPE_CLEAN_PKL, 'rb') as f:
        all_curriculum = pickle.load(f)
    print(f'Loaded POPE-clean curriculum: {POPE_CLEAN_PKL}')
else:
    print('POPE-clean pickle not found — building full curriculum from loaders ...')
    coco_samples = load_coco_detection_samples(config.data_dir)
    refcoco_samples = load_refcoco_samples(config.data_dir, splits=['train'])
    vg_samples = load_visual_genome_samples(
        config.data_dir, max_per_image=config.vg_max_annotations_per_image
    )
    grit_samples = load_grit_samples(
        config.data_dir, shard=0,
        max_images=250000,
        max_boxes_per_image=16,
        use_ref_exps=True,
        min_clip_l14=0.30,
        min_box_frac=0.05,
    )
    vg_samples = vg_samples + grit_samples
    all_curriculum = coco_samples + refcoco_samples + vg_samples

print(f'Total curriculum rows: {len(all_curriculum):,}')
print(f'Unique train images:   {len({s["image_path"] for s in all_curriculum}):,}')

## 3. Resolve VQAv2 test2015 reference paths

Both **test-dev** and **test-standard** draw questions from the same **test2015** image pool. We hash every on-disk `test2015/*.jpg`.

In [ ]:
split_image_ids, union_image_ids = load_vqav2_eval_image_ids(VQAV2_DIR)

vqav2_ref_paths = collect_vqav2_test2015_paths(str(VQAV2_DIR))
assert len(vqav2_ref_paths) > 0, (
    f'No test2015 images under {VQAV2_DIR / "test2015"}. Run Section 0 download first.'
)

# Sanity: map question image_ids -> expected filenames on disk
test2015_dir = VQAV2_DIR / 'test2015'
missing_ids = []
for image_id in sorted(union_image_ids):
    fname = f'COCO_test2015_{image_id:012d}.jpg'
    if not (test2015_dir / fname).exists():
        missing_ids.append(image_id)

print(f'test2015 JPGs on disk: {len(vqav2_ref_paths):,}')
print(f'Unique image_ids in Q JSON union: {len(union_image_ids):,}')
print(f'Question image_ids missing on disk: {len(missing_ids):,}')
if missing_ids[:5]:
    print('First missing image_ids:', missing_ids[:5])

## 4. Clear shared pHash cache (important)

Delete cached hash matrices so the **reference** matrix is rebuilt from **test2015** images (not POPE val2014 or an old scan).

In [ ]:
PHASH_CACHE_FILES = [
    Path(os.environ.get('REVA_DATA_ROOT') or os.path.expanduser('~/reva-data')) / 'ref_packed.npy',
    Path(os.environ.get('REVA_DATA_ROOT') or os.path.expanduser('~/reva-data')) / 'train_packed.npy',
    Path(os.environ.get('REVA_DATA_ROOT') or os.path.expanduser('~/reva-data')) / 'train_valid_indices.npy',
]

for p in PHASH_CACHE_FILES:
    if p.exists():
        p.unlink()
        print(f'Deleted cache: {p}')
    else:
        print(f'No cache (ok): {p}')

print('Ready to hash VQAv2 test2015 reference + curriculum from scratch.')

## 5. Run pHash decontamination (VQAv2 test2015 reference)

Hashes all curriculum rows and all test2015 images; flags pairs with Hamming distance ≤ 4.

In [ ]:
LOG_PATH = DECONTAM_DIR / 'curriculum_vs_vqav2_test2015_phash.json'

log_path = generate_curriculum_decontamination_log(
    curriculum_samples=all_curriculum,
    all_ref_paths=vqav2_ref_paths,
    config=config,
    output_json_path=str(LOG_PATH),
    type='phash',
)

print(f'Log saved: {log_path}')

## 6. Summarise matches

In [ ]:
with open(log_path) as f:
    log_data = json.load(f)

phash_matches = log_data['phash_matches']
corrupt_indices = set(log_data.get('corrupt_indices', []))
unique_removed = set(m['train_idx'] for m in phash_matches)

print(f'pHash match rows: {len(phash_matches):,}')
print(f'Unique curriculum rows flagged: {len(unique_removed):,} / {len(all_curriculum):,}')
print(f'Corrupt/unreadable rows: {len(corrupt_indices):,}')
print(f'Remaining after pHash purge: {len(all_curriculum) - len(unique_removed):,}')

hamming_dist = Counter(m['hamming_distance'] for m in phash_matches)
print('\nHamming distance distribution:')
for d in sorted(hamming_dist):
    print(f'  distance {d}: {hamming_dist[d]:,}')

In [ ]:
removed_sources = Counter(all_curriculum[idx].get('source', 'unknown') for idx in unique_removed)
total_sources = Counter(s.get('source', 'unknown') for s in all_curriculum)

print(f"{'source':<20} {'total':>10} {'removed':>10} {'remaining':>10} {'% removed':>10}")
print('-' * 65)
for source in sorted(total_sources.keys()):
    total = total_sources[source]
    removed = removed_sources.get(source, 0)
    remaining = total - removed
    pct = removed / total * 100 if total else 0
    print(f'{source:<20} {total:>10,} {removed:>10,} {remaining:>10,} {pct:>9.1f}%')

In [ ]:
# How many unique test2015 images triggered at least one curriculum match
ref_triggered = {m['reference_image'] for m in phash_matches}

print(f'test2015 reference images on disk:     {len(vqav2_ref_paths):,}')
print(f'test2015 images with ≥1 pHash match:   {len(ref_triggered):,}')
print(f'test2015 images with no match:           {len(vqav2_ref_paths) - len(ref_triggered):,}')

# Optional: which eval split's image_ids were hit (by filename parsing)
def _image_id_from_test2015_path(p):
    stem = Path(p).stem  # COCO_test2015_000000123456
    return int(stem.split('_')[-1])

triggered_ids = {_image_id_from_test2015_path(p) for p in ref_triggered if 'test2015' in p}
for split_name, ids in split_image_ids.items():
    hit = len(ids & triggered_ids)
    print(f'VQAv2 {split_name} images with ≥1 match: {hit:,} / {len(ids):,}')

## 7. Visual spot-check (optional)

In [ ]:
def show_phash_match(match):
    train_img = Image.open(match['train_image']).convert('RGB')
    ref_img = Image.open(match['reference_image']).convert('RGB')
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(train_img)
    axes[0].set_title(f"Curriculum [{match['train_idx']}]\n{Path(match['train_image']).name}")
    axes[0].axis('off')
    axes[1].imshow(ref_img)
    axes[1].set_title(f"VQAv2 test2015 (Hamming={match['hamming_distance']})\n{Path(match['reference_image']).name}")
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()


if phash_matches:
    exact = [m for m in phash_matches if m['hamming_distance'] == 0]
    borderline = [m for m in phash_matches if m['hamming_distance'] == HAMMING_THRESH]
    print(f'Exact duplicates (distance=0): {len(exact):,}')
    print(f'Borderline (distance={HAMMING_THRESH}): {len(borderline):,}')
    if exact:
        show_phash_match(random.choice(exact))
    if borderline:
        show_phash_match(random.choice(borderline))
else:
    print('No pHash matches — curriculum is clean w.r.t. VQAv2 test2015.')

## 8. Export CSV audit trail

In [ ]:
csv_path = DECONTAM_DIR / 'phash_matches_curriculum_vqav2_report.csv'
fieldnames = ['train_idx', 'train_image', 'reference_image', 'hamming_distance', 'train_source']

with open(csv_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for m in phash_matches:
        writer.writerow({
            'train_idx': m['train_idx'],
            'train_image': m['train_image'],
            'reference_image': m['reference_image'],
            'hamming_distance': m['hamming_distance'],
            'train_source': all_curriculum[m['train_idx']].get('source', 'unknown'),
        })

print(f'Saved {len(phash_matches):,} rows -> {csv_path}')

## 9. Build clean curriculum & save for training

In [ ]:
clean_curriculum = filter_curriculum_from_log(
    curriculum_samples=all_curriculum,
    json_log_path=str(log_path),
)

clean_pkl = DECONTAM_DIR / 'curriculum_pope_vqav2_clean.pkl'
with open(clean_pkl, 'wb') as f:
    pickle.dump(clean_curriculum, f)

print(f'Clean curriculum saved: {clean_pkl}')
print(f'Rows: {len(clean_curriculum):,}')
print(f'Unique images: {len({s["image_path"] for s in clean_curriculum}):,}')

## 10. Load in training notebook

```python
import pickle

with open(Path(os.environ.get('REVA_DECONTAMINATION_ROOT') or os.path.expanduser('~/reva-data/decontamination')) / 'curriculum_pope_vqav2_clean.pkl', 'rb') as f:
    all_curriculum = pickle.load(f)

coco_samples = [s for s in all_curriculum if s.get('source') == 'coco']
refcoco_samples = [s for s in all_curriculum if s.get('source') in ('refcoco', 'refcocog', 'refcoco+')]
vg_samples = [s for s in all_curriculum if s.get('source') in ('visual_genome', 'grit')]
print(f'COCO: {len(coco_samples):,} | RefCOCO: {len(refcoco_samples):,} | VG+GRIT: {len(vg_samples):,}')
```

**Note:** test-dev and test-standard share test2015 images — this pickle protects both. For thesis tables, you can still submit test-dev / test-std separately at eval time; training leakage is blocked at the image level.